# D365 F&O Observability Notebook

Notebook para ejecutar consultas KQL contra Log Analytics / Application Insights y analizar telemetria de Dynamics 365 Finance & Operations.

Bloques principales:
- Discovery de tablas y eventos
- SLI/SLO inicial
- Rendimiento de formularios y slow queries
- Excepciones y batch
- Exportacion a Excel y CSV

In [ ]:
from datetime import timedelta
from pathlib import Path
import os

import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from azure.identity import AzureCliCredential, ChainedTokenCredential, DeviceCodeCredential, ClientSecretCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus
from azure.core.exceptions import HttpResponseError

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 200)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

QUERIES_DIR = PROJECT_ROOT / "queries"
EXPORTS_DIR = PROJECT_ROOT / "exports"
EXPORTS_DIR.mkdir(exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")
WORKSPACE_ID = os.getenv("LAW_WORKSPACE_ID", "").strip()
APPINSIGHTS_RESOURCE_ID = os.getenv("APPINSIGHTS_RESOURCE_ID", "").strip()
APPINSIGHTS_CONNECTION_STRING = os.getenv("APPINSIGHTS_CONNECTION_STRING", "").strip()
QUERY_DAYS = int(os.getenv("QUERY_DAYS", "7"))
AZURE_TENANT_ID = os.getenv("AZURE_TENANT_ID", "").strip()
AZURE_CLIENT_ID = os.getenv("AZURE_CLIENT_ID", "").strip()
AZURE_CLIENT_SECRET = os.getenv("AZURE_CLIENT_SECRET", "").strip()

# Configuracion rapida de ventana de analisis para todo el notebook.
# Cambia este valor al principio y se aplicara a todas las consultas.
ANALYSIS_DAYS = 30
# ANALYSIS_DAYS = QUERY_DAYS

if not WORKSPACE_ID and not APPINSIGHTS_RESOURCE_ID:
    raise ValueError("Falta LAW_WORKSPACE_ID o APPINSIGHTS_RESOURCE_ID en el archivo .env")

print(f"Workspace configurado: {'si' if bool(WORKSPACE_ID) else 'no'}")
print(f"AppInsights resource configurado: {'si' if bool(APPINSIGHTS_RESOURCE_ID) else 'no'}")
print(f"Connection string configurada: {'si' if bool(APPINSIGHTS_CONNECTION_STRING) else 'no'}")
print(f"Query days (.env): {QUERY_DAYS}")
print(f"Analysis days (notebook): {ANALYSIS_DAYS}")
print(f"Tenant ID configurado: {'si' if bool(AZURE_TENANT_ID) else 'no'}")
print(f"Client ID configurado: {'si' if bool(AZURE_CLIENT_ID) else 'no'}")
print(f"Client secret configurado: {'si' if bool(AZURE_CLIENT_SECRET) else 'no'}")
print(f"Auth mode preferred: {'Service Principal' if AZURE_CLIENT_ID and AZURE_CLIENT_SECRET else 'Device Code/Azure CLI'}")

In [ ]:
if AZURE_CLIENT_ID and AZURE_CLIENT_SECRET and AZURE_TENANT_ID:
    credential = ClientSecretCredential(
        tenant_id=AZURE_TENANT_ID,
        client_id=AZURE_CLIENT_ID,
        client_secret=AZURE_CLIENT_SECRET,
    )
    auth_mode = "Service Principal (tenant explicito)"
elif AZURE_CLIENT_ID and AZURE_CLIENT_SECRET:
    # Con SP, tenant_id suele ser obligatorio. Si no viene, usamos credencial interactiva.
    credential = ChainedTokenCredential(
        AzureCliCredential(),
        DeviceCodeCredential(),
    )
    auth_mode = "Azure CLI / Device Code (SP incompleto: falta AZURE_TENANT_ID)"
else:
    # Modo recomendado para CSP delegado: no forzar tenant y dejar que Azure resuelva el contexto.
    credential = ChainedTokenCredential(
        AzureCliCredential(),
        DeviceCodeCredential(),
    )
    auth_mode = "Azure CLI / Device Code (autodetect/delegado)"

client = LogsQueryClient(credential)

print("Cliente de Azure Monitor Logs creado correctamente")
print(f"Modo de autenticacion activo: {auth_mode}")

In [ ]:
# Validacion rapida del target antes de ejecutar baterias de consultas
# Si falla aqui, no merece la pena ejecutar el resto de celdas de analisis.

def validate_workspace_access() -> bool:
    test_query = "print x=1"
    try:
        if APPINSIGHTS_RESOURCE_ID:
            response = client.query_resource(
                resource_id=APPINSIGHTS_RESOURCE_ID,
                query=test_query,
                timespan=timedelta(days=1),
                server_timeout=30,
            )
            target_name = "resource_id"
        else:
            response = client.query_workspace(
                workspace_id=WORKSPACE_ID,
                query=test_query,
                timespan=timedelta(days=1),
                server_timeout=30,
            )
            target_name = "workspace_id"

        if response.status == LogsQueryStatus.SUCCESS:
            print(f"Target accesible ({target_name}) y autenticacion OK")
            return True

        print("Target responde parcialmente")
        print(response.partial_error)
        return True
    except HttpResponseError as err:
        err_text = str(err)
        if "WorkspaceNotFoundError" in err_text:
            print("Workspace no encontrado. Revisa LAW_WORKSPACE_ID en .env")
        elif "PathNotFoundError" in err_text or "ResourceNotFound" in err_text:
            print("Resource ID no encontrado. Revisa APPINSIGHTS_RESOURCE_ID en .env")
        elif "AuthorizationFailed" in err_text or "Forbidden" in err_text:
            print("Sin permisos sobre el target. Solicita rol de lectura")
        else:
            print("Error validando target:")
            print(err_text)
        return False
    except Exception as err:
        print("Error de autenticacion o entorno:")
        print(err)
        return False

workspace_ok = validate_workspace_access()
workspace_ok

In [ ]:
def run_kql(query: str, days: int = ANALYSIS_DAYS, name: str | None = None) -> pd.DataFrame:
    try:
        if name:
            print(f"Ejecutando: {name}")

        if APPINSIGHTS_RESOURCE_ID:
            response = client.query_resource(
                resource_id=APPINSIGHTS_RESOURCE_ID,
                query=query,
                timespan=timedelta(days=days),
                server_timeout=600,
            )
        else:
            response = client.query_workspace(
                workspace_id=WORKSPACE_ID,
                query=query,
                timespan=timedelta(days=days),
                server_timeout=600,
            )

        if response.status == LogsQueryStatus.SUCCESS:
            tables = response.tables
        else:
            print("Consulta con resultado parcial")
            print(response.partial_error)
            tables = response.partial_data

        if not tables:
            return pd.DataFrame()

        table = tables[0]
        df = pd.DataFrame(data=table.rows, columns=table.columns)
        print(f"Filas devueltas: {len(df)}")
        return df

    except HttpResponseError as err:
        print("Error ejecutando KQL")
        print(err)
        return pd.DataFrame()


def load_kql(file_name: str) -> str:
    path = QUERIES_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo KQL: {path}")
    return path.read_text(encoding="utf-8")

In [ ]:
df_tables = run_kql(load_kql("00_discovery_tables.kql"), days=ANALYSIS_DAYS, name="Discovery tables")
df_tables.head(20)

In [ ]:
queries_to_run = {
    "Discovery Tables": "00_discovery_tables.kql",
    "Discovery Events": "01_discovery_events.kql",
    "SLI Executive": "10_sli_executive_summary.kql",
    "Slow Forms": "20_slow_forms.kql",
    "Most Used Forms": "21_most_used_forms.kql",
    "Forms Total Impact": "22_forms_total_impact.kql",
    "Slow Queries": "30_slow_queries.kql",
    "Slow Query Details": "31_slow_queries_details.kql",
    "Exceptions": "40_exceptions.kql",
    "Exceptions Timeseries": "41_exceptions_timeseries.kql",
    "Batch Discovery": "50_batch_discovery.kql",
    "Batch Event Name Discovery": "53_batch_event_name_discovery.kql",
    "Batch Dimension Key Discovery": "54_batch_dimension_key_discovery.kql",
}

results = {}
for label, file_name in queries_to_run.items():
    results[label] = run_kql(load_kql(file_name), days=ANALYSIS_DAYS, name=label)


def _prepare_for_excel(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        # Excel no soporta datetime con timezone; los convertimos a naive UTC.
        if isinstance(out[col].dtype, pd.DatetimeTZDtype):
            out[col] = out[col].dt.tz_convert("UTC").dt.tz_localize(None)
    return out


output_path = EXPORTS_DIR / "d365_fo_observability.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in results.items():
        safe_df = _prepare_for_excel(df)
        safe_df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print(f"Exportado: {output_path}")

# Vista rapida para discovery batch
if "Batch Event Name Discovery" in results and not results["Batch Event Name Discovery"].empty:
    print("\nTop Batch Event Names:")
    display(results["Batch Event Name Discovery"].head(20))

if "Batch Dimension Key Discovery" in results and not results["Batch Dimension Key Discovery"].empty:
    print("\nTop Batch Dimension Keys:")
    display(results["Batch Dimension Key Discovery"].head(30))